# Statistical tests for Fig. 6 (Section 4.4)

- Input: `../param/wilcoxon_holm/participant_mean.csv` created by `result_init_param.ipynb` (the mean of the 50 parameter sets for each participant, n = 18 per condition).
- Wilcoxon signed-rank test (two-sided) for all pairs of parameter ranges, with the p values adjusted by the Holm method in each condition.

In [1]:
import itertools as it

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [2]:
df = pd.read_csv("../param/wilcoxon_holm/participant_mean.csv")
scene_lis = ["office", "supermarket"]
method_lis = ["I-VT", "I-DT", "I-VDT"]
range_lis = [1, 2, 3, 4, 5]
out_dir = "../param/wilcoxon_holm"

In [3]:
def wilcoxon_holm(data):
    """Run the Wilcoxon signed-rank test for all pairs of parameter ranges and adjust the p values with the Holm method."""
    wide = data.pivot(index="participant", columns="range", values="error")
    rows = []
    for a, b in it.combinations(range_lis, 2):
        r = stats.wilcoxon(wide[a], wide[b])
        rows.append({"group1": a, "group2": b, "W": r.statistic, "p_raw": r.pvalue})
    res = pd.DataFrame(rows)
    res["p_adj"] = multipletests(res["p_raw"], method="holm")[1]
    res["sig"] = np.select([res["p_adj"] < 0.01, res["p_adj"] < 0.05], ["**", "*"], "")
    return res

## Comparison of parameter ranges (for each scene and algorithm)

In [4]:
res = []
for scene in scene_lis:
    for method in method_lis:
        r = wilcoxon_holm(df[(df["scene"] == scene) & (df["method"] == method)])
        r.insert(0, "method", method)
        r.insert(0, "scene", scene)
        res.append(r)
fig6_range = pd.concat(res, ignore_index=True)
fig6_range.to_csv(f"{out_dir}/fig6_range.csv", index=False)
fig6_range.round(4)

,scene,method,group1,group2,W,p_raw,p_adj,sig
0,office,I-VT,1,2,84.0,0.9661,1.0000,
1,office,I-VT,1,3,63.0,0.3465,1.0000,
2,office,I-VT,1,4,61.0,0.3038,1.0000,
3,office,I-VT,1,5,59.0,0.2645,1.0000,
4,office,I-VT,2,3,49.0,0.1187,0.7587,
5,office,I-VT,2,4,44.0,0.0737,0.5895,
6,office,I-VT,2,5,41.0,0.0539,0.4847,
7,office,I-VT,3,4,48.0,0.1084,0.7587,
8,office,I-VT,3,5,40.0,0.0483,0.4828,
9,office,I-VT,4,5,49.0,0.1187,0.7587,
